In [1]:
import sys
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import os

############################ real data training ############################
sys.path.append("/home/qiyuanliu/data_filter/Verified-Synthetic-Data/MNIST")

import torch.nn.functional as F
from cvae_model import CVAE, cvae_loss
from torch.utils.data import Subset

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
latent_dim = 20
label_dim = 10
batch_size = 128
epochs = 200
lr = 1e-3
# Load MNIST
transform = transforms.ToTensor()
full_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_dataset = full_dataset
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Initialize model
model = CVAE(latent_dim=latent_dim, label_dim=label_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# One-hot encoding helper
def one_hot(labels, num_classes=10):
    return F.one_hot(labels, num_classes).float()

best_train_loss = float('inf')
patience = 5
trigger_times = 0

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x = x.view(-1, 784).to(device)
        y = one_hot(y).to(device)

        optimizer.zero_grad()
        recon_x, mu, logvar = model(x, y)
        loss = cvae_loss(recon_x, x, mu, logvar)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {avg_loss:.4f}")

    # Early stopping based on training loss
    if avg_loss < best_train_loss:
        best_train_loss = avg_loss
        trigger_times = 0
    else:
        trigger_times += 1
        print(f"EarlyStopping counter: {trigger_times} out of {patience}")
        if trigger_times >= patience:
            print("Early stopping triggered.")
            break
os.chdir("/home/qiyuanliu/data_filter/Verified-Synthetic-Data/MNIST")  
# save the model to model_saved folder
torch.save(model.state_dict(), "model_saved/cvae_mnist_full.pth")
print(f"Model saved to model_saved/cvae_mnist_full.pth")

############################ generate synthetic data ############################

model = CVAE(latent_dim=latent_dim, label_dim=label_dim)
model.load_state_dict(torch.load(f"model_saved/cvae_mnist_full.pth"))
model.eval()

def generate_images_in_batches(model, total_samples, latent_dim, num_classes, batch_size=10000, device='cuda'):
    model.eval()
    generated_images = []
    all_labels = []

    for start in range(0, total_samples, batch_size):
        end = min(start + batch_size, total_samples)
        batch_size_actual = end - start

        # Generate z and y
        z = torch.randn(batch_size_actual, latent_dim).to(device)
        y = torch.arange(num_classes).repeat_interleave(total_samples // num_classes)[start:end]
        y_onehot = F.one_hot(y, num_classes=num_classes).float().to(device)

        with torch.no_grad():
            imgs = model.decode(z, y_onehot).view(-1, 1, 28, 28).cpu()
            generated_images.append(imgs)
            all_labels.append(y)

    images = torch.cat(generated_images, dim=0)
    labels = torch.cat(all_labels, dim=0)
    return images, labels

# large sample size for training
latent_dim = model.latent_dim
device = next(model.parameters()).device
gen_imgs_before_filter,y_before_filter = generate_images_in_batches(
    model=model,
    total_samples=60000,
    latent_dim=latent_dim,
    num_classes=10,
    batch_size=10000,
    device=device
)

Epoch [1/200], Train Loss: 164.3684
Epoch [2/200], Train Loss: 120.3001
Epoch [3/200], Train Loss: 112.5133
Epoch [4/200], Train Loss: 109.0320
Epoch [5/200], Train Loss: 107.0422
Epoch [6/200], Train Loss: 105.6279
Epoch [7/200], Train Loss: 104.5837
Epoch [8/200], Train Loss: 103.7986
Epoch [9/200], Train Loss: 103.1272
Epoch [10/200], Train Loss: 102.6464
Epoch [11/200], Train Loss: 102.1301
Epoch [12/200], Train Loss: 101.6792
Epoch [13/200], Train Loss: 101.3778
Epoch [14/200], Train Loss: 101.0476
Epoch [15/200], Train Loss: 100.7983
Epoch [16/200], Train Loss: 100.5713
Epoch [17/200], Train Loss: 100.2813
Epoch [18/200], Train Loss: 100.0952
Epoch [19/200], Train Loss: 99.9236
Epoch [20/200], Train Loss: 99.6588
Epoch [21/200], Train Loss: 99.5488
Epoch [22/200], Train Loss: 99.4198
Epoch [23/200], Train Loss: 99.2556
Epoch [24/200], Train Loss: 99.1461
Epoch [25/200], Train Loss: 98.9866
Epoch [26/200], Train Loss: 98.8316
Epoch [27/200], Train Loss: 98.7789
Epoch [28/200], Tra

/tmp/ipykernel_1048346/1104007040.py:80: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"model_saved/cvae_mnist_full.pth"))


In [2]:
from FID import calculate_fid_score_2
from torch.utils.data import DataLoader, TensorDataset
transform = transforms.ToTensor()                 # yields (1,28,28) float32
real_ds = datasets.MNIST(root="./data",           # download if not present
                         train=False,
                         transform=transform,
                         download=True)

synthetic_ds = TensorDataset(gen_imgs_before_filter,torch.zeros(len(gen_imgs_before_filter), dtype=torch.long))

fid_value = calculate_fid_score_2(real_ds, synthetic_ds)
print(f"FID(real ↔ synthetic): {fid_value:.2f}")


FID(real ↔ synthetic): 113.18


In [ ]:
import sys
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import os

sample_size = 5000
filter_threshold = 0.5

############################ real data training ############################
sys.path.append("/home/qiyuanliu/data_filter/Verified-Synthetic-Data/MNIST")

import torch.nn.functional as F
from cvae_model import CVAE, cvae_loss
from torch.utils.data import Subset

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
latent_dim = 20
label_dim = 10
batch_size = 128
epochs = 200
lr = 1e-3
# Load MNIST
transform = transforms.ToTensor()
full_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_dataset = Subset(full_dataset, range(sample_size))
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Initialize model
model = CVAE(latent_dim=latent_dim, label_dim=label_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# One-hot encoding helper
def one_hot(labels, num_classes=10):
    return F.one_hot(labels, num_classes).float()

best_train_loss = float('inf')
patience = 50
trigger_times = 0

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x = x.view(-1, 784).to(device)
        y = one_hot(y).to(device)

        optimizer.zero_grad()
        recon_x, mu, logvar = model(x, y)
        loss = cvae_loss(recon_x, x, mu, logvar)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {avg_loss:.4f}")

    # Early stopping based on training loss
    if avg_loss < best_train_loss:
        best_train_loss = avg_loss
        trigger_times = 0
    else:
        trigger_times += 1
        print(f"EarlyStopping counter: {trigger_times} out of {patience}")
        if trigger_times >= patience:
            print("Early stopping triggered.")
            break
os.chdir("/home/qiyuanliu/data_filter/Verified-Synthetic-Data/MNIST")  
# save the model to model_saved folder
torch.save(model.state_dict(), f"model_saved/cvae_mnist_{sample_size}.pth")
print(f"Model saved to model_saved/cvae_mnist_{sample_size}.pth")

############################ generate synthetic data ############################

model = CVAE(latent_dim=latent_dim, label_dim=label_dim)
model.load_state_dict(torch.load(f"model_saved/cvae_mnist_{sample_size}.pth"))
model.eval()

def generate_images_in_batches(model, total_samples, latent_dim, num_classes, batch_size=10000, device='cuda'):
    model.eval()
    generated_images = []
    all_labels = []

    for start in range(0, total_samples, batch_size):
        end = min(start + batch_size, total_samples)
        batch_size_actual = end - start

        # Generate z and y
        z = torch.randn(batch_size_actual, latent_dim).to(device)
        y = torch.arange(num_classes).repeat_interleave(total_samples // num_classes)[start:end]
        y_onehot = F.one_hot(y, num_classes=num_classes).float().to(device)

        with torch.no_grad():
            imgs = model.decode(z, y_onehot).view(-1, 1, 28, 28).cpu()
            generated_images.append(imgs)
            all_labels.append(y)

    images = torch.cat(generated_images, dim=0)
    labels = torch.cat(all_labels, dim=0)
    return images, labels

# large sample size for training
latent_dim = model.latent_dim
device = next(model.parameters()).device
gen_imgs_before_filter,y_before_filter = generate_images_in_batches(
    model=model,
    total_samples=6000000,
    latent_dim=latent_dim,
    num_classes=10,
    batch_size=10000,
    device=device
)

#save_path = f"data_saved/synthetic_mnist_cvae_{sample_size}.pt"
#torch.save({
#    'images': gen_imgs,    # Tensor [6000000, 1, 28, 28]
#    'labels': y            # Tensor [6000000]
#}, save_path)

# smaller sample size for evaluation
gen_imgs,y = generate_images_in_batches(
    model=model,
    total_samples=6000,
    latent_dim=latent_dim,
    num_classes=10,
    batch_size=10000,
    device=device
)

save_path = f"data_saved/synthetic_mnist_cvae_{sample_size}_2.pt"
torch.save({
    'images': gen_imgs,    # Tensor [6000, 1, 28, 28]
    'labels': y            # Tensor [6000]
}, save_path)

############################ filter synthetic data ############################

from discriminator import Discriminator
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
D = Discriminator().to(device)
D.load_state_dict(torch.load("model_saved/discriminator_mnist_cvae_2.pth"))
D.eval()

#data = torch.load(f"data_saved/synthetic_mnist_cvae_{sample_size}.pt")
#synthetic_images = data['images']  

synthetic_loader = DataLoader(gen_imgs_before_filter, batch_size=512)

all_probs = []

with torch.no_grad():
    for batch in synthetic_loader:
        batch = batch.to(device)
        probs = D(batch)  # [batch_size, 1], already sigmoid activated
        all_probs.append(probs.cpu())

all_probs = torch.cat(all_probs, dim=0)
# Flatten probs to shape [N]
probs = all_probs.squeeze(1)

# Load images and labels
images = gen_imgs_before_filter#data['images']      # [N, 1, 28, 28]
labels = y_before_filter #data['labels']      # [N]
# Create mask for p > filter_threshold
mask = probs > filter_threshold

# Apply mask
filtered_images = images[mask]
filtered_labels = labels[mask]

print(f"Selected {filtered_images.shape[0]} samples with p > {filter_threshold}")
# Save to file
torch.save({
    'images': filtered_images,
    'labels': filtered_labels
}, f"data_saved/synthetic_mnist_filtered_pgt{filter_threshold}_{sample_size}.pt")


############################ synthetic data retraining ############################

from torch.utils.data import TensorDataset

images = filtered_images  # shape: [N, 1, 28, 28]
labels = filtered_labels  # shape: [N]

print(f"Loaded {images.shape[0]} filtered synthetic samples")

# Preprocess: flatten images and convert labels to one-hot
images = images.view(-1, 784)  # flatten to [N, 784]

# Create dataset and dataloader
dataset = TensorDataset(images, labels)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialize model
model = CVAE(latent_dim=latent_dim, label_dim=label_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

best_train_loss = float('inf')
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x = x.view(-1, 784).to(device)
        y = one_hot(y).to(device)

        optimizer.zero_grad()
        recon_x, mu, logvar = model(x, y)
        loss = cvae_loss(recon_x, x, mu, logvar)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {avg_loss:.4f}")

    # Early stopping based on training loss
    if avg_loss < best_train_loss:
        best_train_loss = avg_loss
        trigger_times = 0
    else:
        trigger_times += 1
        print(f"EarlyStopping counter: {trigger_times} out of {patience}")
        if trigger_times >= patience:
            print("Early stopping triggered.")
            break

# save the model to model_saved folder
torch.save(model.state_dict(), f"model_saved/cvae_mnist_filtered_synthetic_data_{sample_size}.pth")

# use the new model to generate synthetic data for evaluation
model.eval()

n_per_class = 60000
num_classes = 10
total_samples = n_per_class * num_classes
latent_dim = model.latent_dim
device = next(model.parameters()).device

z = torch.randn(total_samples, latent_dim).to(device)
y = torch.arange(num_classes).repeat_interleave(n_per_class)
y_onehot = F.one_hot(y, num_classes=num_classes).float().to(device)

with torch.no_grad():
    gen_imgs = model.decode(z, y_onehot).view(-1, 1, 28, 28).cpu()  # shape: [60000, 1, 28, 28]

# save the generated images and labels
save_path = f"data_saved/synthetic_mnist_cvae_filtered_synthetic_model_generated_data_{sample_size}.pt"
torch.save({"images": gen_imgs, "labels": y}, save_path)

############################ Model Evaluation ############################

# FID
from FID import calculate_fid_score

transform = transforms.ToTensor()

real_ds = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
### Synthetic dataset
synthetic = torch.load(f"data_saved/synthetic_mnist_cvae_{sample_size}_2.pt")
synthetic_ds = TensorDataset(synthetic['images'], torch.zeros(len(synthetic['images'])))
fid_value = calculate_fid_score(real_ds, synthetic_ds)
print(f"FID Score(real data and synthetic data): {fid_value:.2f}")

## filtered synthetic data
synthetic = torch.load(f"data_saved/synthetic_mnist_filtered_pgt{filter_threshold}_{sample_size}.pt")
synthetic_ds = TensorDataset(synthetic['images'], torch.zeros(len(synthetic['images'])))
fid_value = calculate_fid_score(real_ds, synthetic_ds)
print(f"FID Score(real data and filtered synthetic data): {fid_value:.2f}")

# Synthetic dataset
synthetic = torch.load(f"data_saved/synthetic_mnist_cvae_filtered_synthetic_model_generated_data_{sample_size}.pt")
synthetic_ds = TensorDataset(synthetic['images'], torch.zeros(len(synthetic['images'])))
fid_value = calculate_fid_score(real_ds, synthetic_ds)
print(f"FID Score(real data and model 2 synthetic data): {fid_value:.2f}")

# Reconstruction Loss

# Load model
model = CVAE(latent_dim=20, label_dim=10).to(device)
model.load_state_dict(torch.load(f"model_saved/cvae_mnist_{sample_size}.pth"))
model.eval()

# Load test set
test_dataset = datasets.MNIST(root="./data", train=False, transform=transforms.ToTensor())
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# Evaluation
total_loss = 0
total_recon_loss = 0
total_kl = 0
num_samples = 0

with torch.no_grad():
    for x, y in test_loader:
        x = x.view(-1, 784).to(device)
        y = F.one_hot(y, num_classes=10).float().to(device)

        recon_x, mu, logvar = model(x, y)
        BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
        KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        loss = BCE + KLD

        total_loss += loss.item()
        total_recon_loss += BCE.item()
        total_kl += KLD.item()
        num_samples += x.size(0)

print(f"Test Set Results(model1):")
print(f"  Avg CVAE Loss: {total_loss / num_samples:.4f}")
print(f"  Avg Reconstruction (BCE) Loss: {total_recon_loss / num_samples:.4f}")
print(f"  Avg KL Divergence: {total_kl / num_samples:.4f}")

# get the loss of synthetic model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CVAE(latent_dim=latent_dim, label_dim=label_dim).to(device)
model.load_state_dict(torch.load(f"model_saved/cvae_mnist_filtered_synthetic_data_{sample_size}.pth"))
model.eval()

# Load test set
test_dataset = datasets.MNIST(root="./data", train=False, transform=transforms.ToTensor())
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# Evaluation
total_loss = 0
total_recon_loss = 0
total_kl = 0
num_samples = 0

with torch.no_grad():
    for x, y in test_loader:
        x = x.view(-1, 784).to(device)
        y = F.one_hot(y, num_classes=10).float().to(device)

        recon_x, mu, logvar = model(x, y)
        BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
        KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        loss = BCE + KLD

        total_loss += loss.item()
        total_recon_loss += BCE.item()
        total_kl += KLD.item()
        num_samples += x.size(0)

print(f"Test Set Results(model2):")
print(f"  Avg CVAE Loss: {total_loss / num_samples:.4f}")
print(f"  Avg Reconstruction (BCE) Loss: {total_recon_loss / num_samples:.4f}")
print(f"  Avg KL Divergence: {total_kl / num_samples:.4f}")

Epoch [1/200], Train Loss: 289.2158
Epoch [2/200], Train Loss: 202.6939
Epoch [3/200], Train Loss: 179.6633
Epoch [4/200], Train Loss: 162.0263
Epoch [5/200], Train Loss: 152.7175
Epoch [6/200], Train Loss: 146.2386
Epoch [7/200], Train Loss: 140.8810
Epoch [8/200], Train Loss: 136.1365
Epoch [9/200], Train Loss: 132.3334
Epoch [10/200], Train Loss: 129.0440
Epoch [11/200], Train Loss: 126.9454
Epoch [12/200], Train Loss: 124.6291
Epoch [13/200], Train Loss: 122.8178
Epoch [14/200], Train Loss: 121.1568
Epoch [15/200], Train Loss: 119.8952
Epoch [16/200], Train Loss: 118.6535
Epoch [17/200], Train Loss: 117.3821
Epoch [18/200], Train Loss: 116.2011
Epoch [19/200], Train Loss: 115.5105
Epoch [20/200], Train Loss: 114.7104
Epoch [21/200], Train Loss: 113.9667
Epoch [22/200], Train Loss: 113.3592
Epoch [23/200], Train Loss: 112.5861
Epoch [24/200], Train Loss: 112.1151
Epoch [25/200], Train Loss: 111.4629
Epoch [26/200], Train Loss: 110.9937
Epoch [27/200], Train Loss: 110.6352
Epoch [28/

/tmp/ipykernel_1256767/205434050.py:83: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"model_saved/cvae_mnist_{sample_size}.pth"))
/tmp/ipy

Selected 57425 samples with p > 0.5
Loaded 57425 filtered synthetic samples
Epoch [1/200], Train Loss: 146.1120
Epoch [2/200], Train Loss: 107.3864
Epoch [3/200], Train Loss: 101.2209
Epoch [4/200], Train Loss: 98.2969
Epoch [5/200], Train Loss: 96.8542
Epoch [6/200], Train Loss: 95.8540
Epoch [7/200], Train Loss: 95.2023
Epoch [8/200], Train Loss: 94.6796
Epoch [9/200], Train Loss: 94.2917
Epoch [10/200], Train Loss: 94.0005
Epoch [11/200], Train Loss: 93.7045
Epoch [12/200], Train Loss: 93.5315
Epoch [13/200], Train Loss: 93.3751
Epoch [14/200], Train Loss: 93.1906
Epoch [15/200], Train Loss: 93.0695
Epoch [16/200], Train Loss: 92.9931
Epoch [17/200], Train Loss: 92.8923
Epoch [18/200], Train Loss: 92.7764
Epoch [19/200], Train Loss: 92.6725
Epoch [20/200], Train Loss: 92.6251
Epoch [21/200], Train Loss: 92.5513
Epoch [22/200], Train Loss: 92.5046
Epoch [23/200], Train Loss: 92.4017
Epoch [24/200], Train Loss: 92.4138
EarlyStopping counter: 1 out of 5
Epoch [25/200], Train Loss: 92.3

/tmp/ipykernel_1256767/205434050.py:269: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  synthetic = torch.load(f"data_saved/synthetic_mnist_cvae_{sample_size}_2.pt")
Extracti

FID Score(real data and synthetic data): 0.05


...

FID Score(real data and model 2 synthetic data): 0.03


/tmp/ipykernel_1256767/205434050.py:290: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"model_saved/cvae_mnist_{sample_size}.pth"))


Test Set Results(model1):
  Avg CVAE Loss: 106.7203
  Avg Reconstruction (BCE) Loss: 84.9539
  Avg KL Divergence: 21.7664


/tmp/ipykernel_1256767/205434050.py:327: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"model_saved/cvae_mnist_filtered_synthetic_data_{sam

Test Set Results(model2):
  Avg CVAE Loss: 110.1210
  Avg Reconstruction (BCE) Loss: 88.5636
  Avg KL Divergence: 21.5574


In [7]:
def evaluate_cvae(model, dataloader, device, num_classes=10):
    """
    Evaluate a CVAE model on a given dataloader.

    Args:
        model: The trained CVAE model.
        dataloader: PyTorch DataLoader for evaluation.
        device: torch.device('cuda') or torch.device('cpu').
        num_classes: Number of label classes (default=10).

    Returns:
        dict with average total loss, reconstruction loss (BCE), and KL divergence.
    """
    model.eval()
    total_loss = 0.0
    total_recon_loss = 0.0
    total_kl = 0.0
    num_samples = 0

    with torch.no_grad():
        for x, y in dataloader:
            x = x.view(-1, 784).to(device)
            y = F.one_hot(y, num_classes=num_classes).float().to(device)

            recon_x, mu, logvar = model(x, y)
            BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
            KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
            loss = BCE + KLD

            total_loss += loss.item()
            total_recon_loss += BCE.item()
            total_kl += KLD.item()
            num_samples += x.size(0)

    return {
        "avg_total_loss": total_loss / num_samples,
        "avg_recon_loss": total_recon_loss / num_samples,
        "avg_kl_divergence": total_kl / num_samples
    }

test_dataset = datasets.MNIST(root="./data", train=False, transform=transforms.ToTensor())
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

model1 = CVAE(latent_dim=latent_dim, label_dim=label_dim).to(device)
model1.load_state_dict(torch.load(f"model_saved/cvae_mnist_{sample_size}.pth"))
model1_score = evaluate_cvae(model1, test_loader, device)

model2 = CVAE(latent_dim=latent_dim, label_dim=label_dim).to(device)
model2.load_state_dict(torch.load(f"model_saved/cvae_mnist_filtered_synthetic_data_{sample_size}.pth"))
model2_score = evaluate_cvae(model2, test_loader, device)

model1_score, model2_score

/tmp/ipykernel_1256767/4138888054.py:45: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model1.load_state_dict(torch.load(f"model_saved/cvae_mnist_{sample_size}.pth"))
/tmp/i

({'avg_total_loss': 106.80625124511718,
  'avg_recon_loss': 85.03983623046875,
  'avg_kl_divergence': 21.766415252685547},
 {'avg_total_loss': 110.19495974121094,
  'avg_recon_loss': 88.63756151123047,
  'avg_kl_divergence': 21.557398333740235})

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CVAE(latent_dim=latent_dim, label_dim=label_dim).to(device)
model.load_state_dict(torch.load(f"model_saved/cvae_mnist_filtered_synthetic_data_{sample_size}.pth"))
model.eval()

# Load test set
test_dataset = datasets.MNIST(root="./data", train=False, transform=transforms.ToTensor())
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# Evaluation
total_loss = 0
total_recon_loss = 0
total_kl = 0
num_samples = 0

with torch.no_grad():
    for x, y in test_loader:
        x = x.view(-1, 784).to(device)
        y = F.one_hot(y, num_classes=10).float().to(device)

        recon_x, mu, logvar = model(x, y)
        BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
        KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        loss = BCE + KLD

        total_loss += loss.item()
        total_recon_loss += BCE.item()
        total_kl += KLD.item()
        num_samples += x.size(0)

print(f"Test Set Results(model2):")
print(f"  Avg CVAE Loss: {total_loss / num_samples:.4f}")
print(f"  Avg Reconstruction (BCE) Loss: {total_recon_loss / num_samples:.4f}")
print(f"  Avg KL Divergence: {total_kl / num_samples:.4f}")

/tmp/ipykernel_1256767/3831585684.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"model_saved/cvae_mnist_filtered_synthetic_data_{samp

Test Set Results(model2):
  Avg CVAE Loss: 110.1666
  Avg Reconstruction (BCE) Loss: 88.6092
  Avg KL Divergence: 21.5574


In [9]:
evaluate_cvae(model2, test_loader, device)

{'avg_total_loss': 110.1193423828125,
 'avg_recon_loss': 88.56194404296875,
 'avg_kl_divergence': 21.557398333740235}

In [ ]:
import sys
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import os
import numpy as np
import random

sample_size = 5000
filter_threshold = 0.2

############################ real data training ############################
sys.path.append("/home/qiyuanliu/data_filter/Verified-Synthetic-Data/MNIST")
seed = 0
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

import torch.nn.functional as F
from cvae_model import CVAE, cvae_loss
from torch.utils.data import Subset

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
latent_dim = 20
label_dim = 10
batch_size = 128
epochs = 200
lr = 1e-3
# Load MNIST
transform = transforms.ToTensor()
full_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_dataset = Subset(full_dataset, range(sample_size))
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Initialize model
model = CVAE(latent_dim=latent_dim, label_dim=label_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# One-hot encoding helper
def one_hot(labels, num_classes=10):
    return F.one_hot(labels, num_classes).float()

best_train_loss = float('inf')
patience = 50

trigger_times = 0

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x = x.view(-1, 784).to(device)
        y = one_hot(y).to(device)

        optimizer.zero_grad()
        recon_x, mu, logvar = model(x, y)
        loss = cvae_loss(recon_x, x, mu, logvar)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {avg_loss:.4f}")

    # Early stopping based on training loss
    if avg_loss < best_train_loss:
        best_train_loss = avg_loss
        trigger_times = 0
    else:
        trigger_times += 1
        print(f"EarlyStopping counter: {trigger_times} out of {patience}")
        if trigger_times >= patience:
            print("Early stopping triggered.")
            break
os.chdir("/home/qiyuanliu/data_filter/Verified-Synthetic-Data/MNIST")  
# save the model to model_saved folder
torch.save(model.state_dict(), f"model_saved/cvae_mnist_{sample_size}.pth")
print(f"Model saved to model_saved/cvae_mnist_{sample_size}.pth")

############################ generate synthetic data ############################

model = CVAE(latent_dim=latent_dim, label_dim=label_dim)
model.load_state_dict(torch.load(f"model_saved/cvae_mnist_{sample_size}.pth"))
model.eval()

def generate_images_in_batches(model, total_samples, latent_dim, num_classes, batch_size=10000, device='cuda'):
    model.eval()
    generated_images = []
    all_labels = []

    for start in range(0, total_samples, batch_size):
        end = min(start + batch_size, total_samples)
        batch_size_actual = end - start

        # Generate z and y
        z = torch.randn(batch_size_actual, latent_dim).to(device)
        y = torch.arange(num_classes).repeat_interleave(total_samples // num_classes)[start:end]
        y_onehot = F.one_hot(y, num_classes=num_classes).float().to(device)

        with torch.no_grad():
            imgs = model.decode(z, y_onehot).view(-1, 1, 28, 28).cpu()
            generated_images.append(imgs)
            all_labels.append(y)

    images = torch.cat(generated_images, dim=0)
    labels = torch.cat(all_labels, dim=0)
    return images, labels

# large sample size for training
latent_dim = model.latent_dim
device = next(model.parameters()).device
gen_imgs_before_filter,y_before_filter = generate_images_in_batches(
    model=model,
    total_samples=60000,#6000000,
    latent_dim=latent_dim,
    num_classes=10,
    batch_size=10000,
    device=device
)

#save_path = f"data_saved/synthetic_mnist_cvae_{sample_size}.pt"
#torch.save({
#    'images': gen_imgs,    # Tensor [6000000, 1, 28, 28]
#    'labels': y            # Tensor [6000000]
#}, save_path)

# smaller sample size for evaluation
gen_imgs,y = generate_images_in_batches(
    model=model,
    total_samples=6000,
    latent_dim=latent_dim,
    num_classes=10,
    batch_size=10000,
    device=device
)

save_path = f"data_saved/synthetic_mnist_cvae_{sample_size}_2.pt"
torch.save({
    'images': gen_imgs,    # Tensor [6000, 1, 28, 28]
    'labels': y            # Tensor [6000]
}, save_path)

############################ filter synthetic data ############################

from discriminator import Discriminator
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
D = Discriminator().to(device)
D.load_state_dict(torch.load("model_saved/discriminator_mnist_cvae_2.pth"))
D.eval()

#data = torch.load(f"data_saved/synthetic_mnist_cvae_{sample_size}.pt")
#synthetic_images = data['images']  

synthetic_loader = DataLoader(gen_imgs_before_filter, batch_size=512)

all_probs = []

with torch.no_grad():
    for batch in synthetic_loader:
        batch = batch.to(device)
        probs = D(batch)  # [batch_size, 1], already sigmoid activated
        all_probs.append(probs.cpu())

all_probs = torch.cat(all_probs, dim=0)

"""
# Flatten probs to shape [N]
probs = all_probs.squeeze(1)

# Load images and labels
images = gen_imgs_before_filter#data['images']      # [N, 1, 28, 28]
labels = y_before_filter #data['labels']      # [N]
# Create mask for p > filter_threshold
mask = probs > filter_threshold

# Apply mask
filtered_images = images[mask]
filtered_labels = labels[mask]
"""
probs = all_probs.squeeze(1)  # [N]
images = gen_imgs_before_filter
labels = y_before_filter

keep_ratio = 0.8
threshold = torch.quantile(probs, 1 - keep_ratio)  
mask = probs >= threshold
filtered_images = images[mask]
filtered_labels = labels[mask]

print(f"Selected {filtered_images.shape[0]} samples with p > {filter_threshold}")
# Save to file
torch.save({
    'images': filtered_images,
    'labels': filtered_labels
}, f"data_saved/synthetic_mnist_filtered_pgt{filter_threshold}_{sample_size}.pt")


############################ synthetic data retraining ############################

from torch.utils.data import TensorDataset

images = filtered_images  # shape: [N, 1, 28, 28]
labels = filtered_labels  # shape: [N]

print(f"Loaded {images.shape[0]} filtered synthetic samples")

# Preprocess: flatten images and convert labels to one-hot
images = images.view(-1, 784)  # flatten to [N, 784]

# Create dataset and dataloader
dataset = TensorDataset(images, labels)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialize model
model = CVAE(latent_dim=latent_dim, label_dim=label_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

best_train_loss = float('inf')
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x = x.view(-1, 784).to(device)
        y = one_hot(y).to(device)

        optimizer.zero_grad()
        recon_x, mu, logvar = model(x, y)
        loss = cvae_loss(recon_x, x, mu, logvar)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {avg_loss:.4f}")

    # Early stopping based on training loss
    if avg_loss < best_train_loss:
        best_train_loss = avg_loss
        trigger_times = 0
    else:
        trigger_times += 1
        print(f"EarlyStopping counter: {trigger_times} out of {patience}")
        if trigger_times >= patience:
            print("Early stopping triggered.")
            break

# save the model to model_saved folder
torch.save(model.state_dict(), f"model_saved/cvae_mnist_filtered_synthetic_data_{sample_size}.pth")

# use the new model to generate synthetic data for evaluation
model.eval()

n_per_class = 60000
num_classes = 10
total_samples = n_per_class * num_classes
latent_dim = model.latent_dim
device = next(model.parameters()).device

z = torch.randn(total_samples, latent_dim).to(device)
y = torch.arange(num_classes).repeat_interleave(n_per_class)
y_onehot = F.one_hot(y, num_classes=num_classes).float().to(device)

with torch.no_grad():
    gen_imgs = model.decode(z, y_onehot).view(-1, 1, 28, 28).cpu()  # shape: [60000, 1, 28, 28]

# save the generated images and labels
save_path = f"data_saved/synthetic_mnist_cvae_filtered_synthetic_model_generated_data_{sample_size}.pt"
torch.save({"images": gen_imgs, "labels": y}, save_path)

############################ Model Evaluation ############################
"""
# FID
from FID import calculate_fid_score

transform = transforms.ToTensor()

real_ds = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
### Synthetic dataset
synthetic = torch.load(f"data_saved/synthetic_mnist_cvae_{sample_size}_2.pt")
synthetic_ds = TensorDataset(synthetic['images'], torch.zeros(len(synthetic['images'])))
fid_value = calculate_fid_score(real_ds, synthetic_ds)
print(f"FID Score(real data and synthetic data): {fid_value:.2f}")

## filtered synthetic data
synthetic = torch.load(f"data_saved/synthetic_mnist_filtered_pgt{filter_threshold}_{sample_size}.pt")
synthetic_ds = TensorDataset(synthetic['images'], torch.zeros(len(synthetic['images'])))
fid_value = calculate_fid_score(real_ds, synthetic_ds)
print(f"FID Score(real data and filtered synthetic data): {fid_value:.2f}")

# Synthetic dataset
synthetic = torch.load(f"data_saved/synthetic_mnist_cvae_filtered_synthetic_model_generated_data_{sample_size}.pt")
synthetic_ds = TensorDataset(synthetic['images'], torch.zeros(len(synthetic['images'])))
fid_value = calculate_fid_score(real_ds, synthetic_ds)
print(f"FID Score(real data and model 2 synthetic data): {fid_value:.2f}")
"""


def evaluate_cvae(model, dataloader, device, num_classes=10):
    """
    Evaluate a CVAE model on a given dataloader.

    Args:
        model: The trained CVAE model.
        dataloader: PyTorch DataLoader for evaluation.
        device: torch.device('cuda') or torch.device('cpu').
        num_classes: Number of label classes (default=10).

    Returns:
        dict with average total loss, reconstruction loss (BCE), and KL divergence.
    """
    model.eval()
    total_loss = 0.0
    total_recon_loss = 0.0
    total_kl = 0.0
    num_samples = 0

    with torch.no_grad():
        for x, y in dataloader:
            x = x.view(-1, 784).to(device)
            y = F.one_hot(y, num_classes=num_classes).float().to(device)

            recon_x, mu, logvar = model(x, y)
            BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
            KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
            loss = BCE + KLD

            total_loss += loss.item()
            total_recon_loss += BCE.item()
            total_kl += KLD.item()
            num_samples += x.size(0)

    return {
        "avg_total_loss": total_loss / num_samples,
        "avg_recon_loss": total_recon_loss / num_samples,
        "avg_kl_divergence": total_kl / num_samples
    }

test_dataset = datasets.MNIST(root="./data", train=False, transform=transforms.ToTensor())
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

model1 = CVAE(latent_dim=latent_dim, label_dim=label_dim).to(device)
model1.load_state_dict(torch.load(f"model_saved/cvae_mnist_{sample_size}.pth"))
model1_score = evaluate_cvae(model1, test_loader, device)

model2 = CVAE(latent_dim=latent_dim, label_dim=label_dim).to(device)
model2.load_state_dict(torch.load(f"model_saved/cvae_mnist_filtered_synthetic_data_{sample_size}.pth"))
model2_score = evaluate_cvae(model2, test_loader, device)

# save the scores tp compare_result
compare_result = {
    "model1": model1_score,
    "model2": model2_score
}
import joblib
joblib.dump(compare_result, f"compare_result/cvae_mnist_{sample_size}_compare_result.pkl")

# Print the scores
print(f"Model 1 (original) scores: {model1_score}")
print(f"Model 2 (filtered synthetic) scores: {model2_score}")


Epoch [1/200], Train Loss: 289.2108
Epoch [2/200], Train Loss: 203.2987
Epoch [3/200], Train Loss: 180.5909
Epoch [4/200], Train Loss: 163.6566
Epoch [5/200], Train Loss: 153.5960
Epoch [6/200], Train Loss: 147.1924
Epoch [7/200], Train Loss: 142.2007
Epoch [8/200], Train Loss: 138.1660
Epoch [9/200], Train Loss: 135.0459
Epoch [10/200], Train Loss: 131.6477
Epoch [11/200], Train Loss: 128.9189
Epoch [12/200], Train Loss: 126.7101
Epoch [13/200], Train Loss: 124.5109
Epoch [14/200], Train Loss: 122.8315
Epoch [15/200], Train Loss: 121.3991
Epoch [16/200], Train Loss: 120.2449
Epoch [17/200], Train Loss: 119.0302
Epoch [18/200], Train Loss: 117.9980
Epoch [19/200], Train Loss: 116.8348
Epoch [20/200], Train Loss: 115.9087
Epoch [21/200], Train Loss: 115.1816
Epoch [22/200], Train Loss: 114.4127
Epoch [23/200], Train Loss: 113.7345
Epoch [24/200], Train Loss: 112.9594
Epoch [25/200], Train Loss: 112.5227
Epoch [26/200], Train Loss: 111.9417
Epoch [27/200], Train Loss: 111.3447
Epoch [28/

/tmp/ipykernel_1863339/3238875182.py:90: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"model_saved/cvae_mnist_{sample_size}.pth"))
/tmp/ip

Selected 48000 samples with p > 0.2
Loaded 48000 filtered synthetic samples
Epoch [1/200], Train Loss: 168.3174
Epoch [2/200], Train Loss: 129.6182
Epoch [3/200], Train Loss: 123.6844
Epoch [4/200], Train Loss: 121.1855
Epoch [5/200], Train Loss: 119.6844
Epoch [6/200], Train Loss: 118.7426
Epoch [7/200], Train Loss: 118.0453
Epoch [8/200], Train Loss: 117.4963
Epoch [9/200], Train Loss: 117.1585
Epoch [10/200], Train Loss: 116.8314
Epoch [11/200], Train Loss: 116.5756
Epoch [12/200], Train Loss: 116.3625
Epoch [13/200], Train Loss: 116.2040
Epoch [14/200], Train Loss: 116.0494
Epoch [15/200], Train Loss: 115.8930
Epoch [16/200], Train Loss: 115.8367
Epoch [17/200], Train Loss: 115.6987
Epoch [18/200], Train Loss: 115.6418
Epoch [19/200], Train Loss: 115.5679
Epoch [20/200], Train Loss: 115.5036
Epoch [21/200], Train Loss: 115.4511
Epoch [22/200], Train Loss: 115.3965
Epoch [23/200], Train Loss: 115.3391
Epoch [24/200], Train Loss: 115.3236
Epoch [25/200], Train Loss: 115.2796
Epoch [2

/tmp/ipykernel_1863339/3238875182.py:351: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model1.load_state_dict(torch.load(f"model_saved/cvae_mnist_{sample_size}.pth"))
/tmp/

Model 1 (original) scores: {'avg_total_loss': 106.60475462646484, 'avg_recon_loss': 84.78307877197265, 'avg_kl_divergence': 21.821676419067384}
Model 2 (filtered synthetic) scores: {'avg_total_loss': 107.47667077636719, 'avg_recon_loss': 87.49166905517578, 'avg_kl_divergence': 19.985001821899413}


In [3]:
probs = all_probs.squeeze(1)  # [N]
images = gen_imgs_before_filter
labels = y_before_filter

keep_ratio = 0.5
threshold = torch.quantile(probs, 1 - keep_ratio)  
mask = probs >= threshold
filtered_images = images[mask]
filtered_labels = labels[mask]

print(f"Selected {filtered_images.shape[0]} samples with p > {filter_threshold}")
# Save to file
torch.save({
    'images': filtered_images,
    'labels': filtered_labels
}, f"data_saved/synthetic_mnist_filtered_pgt{filter_threshold}_{sample_size}.pt")


############################ synthetic data retraining ############################

from torch.utils.data import TensorDataset

images = filtered_images  # shape: [N, 1, 28, 28]
labels = filtered_labels  # shape: [N]

print(f"Loaded {images.shape[0]} filtered synthetic samples")

# Preprocess: flatten images and convert labels to one-hot
images = images.view(-1, 784)  # flatten to [N, 784]

# Create dataset and dataloader
dataset = TensorDataset(images, labels)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialize model
model = CVAE(latent_dim=latent_dim, label_dim=label_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

best_train_loss = float('inf')
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x = x.view(-1, 784).to(device)
        y = one_hot(y).to(device)

        optimizer.zero_grad()
        recon_x, mu, logvar = model(x, y)
        loss = cvae_loss(recon_x, x, mu, logvar)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {avg_loss:.4f}")

    # Early stopping based on training loss
    if avg_loss < best_train_loss:
        best_train_loss = avg_loss
        trigger_times = 0
    else:
        trigger_times += 1
        print(f"EarlyStopping counter: {trigger_times} out of {patience}")
        if trigger_times >= patience:
            print("Early stopping triggered.")
            break

# save the model to model_saved folder
torch.save(model.state_dict(), f"model_saved/cvae_mnist_filtered_synthetic_data_{sample_size}.pth")

# use the new model to generate synthetic data for evaluation
model.eval()

n_per_class = 60000
num_classes = 10
total_samples = n_per_class * num_classes
latent_dim = model.latent_dim
device = next(model.parameters()).device

z = torch.randn(total_samples, latent_dim).to(device)
y = torch.arange(num_classes).repeat_interleave(n_per_class)
y_onehot = F.one_hot(y, num_classes=num_classes).float().to(device)

with torch.no_grad():
    gen_imgs = model.decode(z, y_onehot).view(-1, 1, 28, 28).cpu()  # shape: [60000, 1, 28, 28]

# save the generated images and labels
save_path = f"data_saved/synthetic_mnist_cvae_filtered_synthetic_model_generated_data_{sample_size}.pt"
torch.save({"images": gen_imgs, "labels": y}, save_path)

############################ Model Evaluation ############################
"""
# FID
from FID import calculate_fid_score

transform = transforms.ToTensor()

real_ds = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
### Synthetic dataset
synthetic = torch.load(f"data_saved/synthetic_mnist_cvae_{sample_size}_2.pt")
synthetic_ds = TensorDataset(synthetic['images'], torch.zeros(len(synthetic['images'])))
fid_value = calculate_fid_score(real_ds, synthetic_ds)
print(f"FID Score(real data and synthetic data): {fid_value:.2f}")

## filtered synthetic data
synthetic = torch.load(f"data_saved/synthetic_mnist_filtered_pgt{filter_threshold}_{sample_size}.pt")
synthetic_ds = TensorDataset(synthetic['images'], torch.zeros(len(synthetic['images'])))
fid_value = calculate_fid_score(real_ds, synthetic_ds)
print(f"FID Score(real data and filtered synthetic data): {fid_value:.2f}")

# Synthetic dataset
synthetic = torch.load(f"data_saved/synthetic_mnist_cvae_filtered_synthetic_model_generated_data_{sample_size}.pt")
synthetic_ds = TensorDataset(synthetic['images'], torch.zeros(len(synthetic['images'])))
fid_value = calculate_fid_score(real_ds, synthetic_ds)
print(f"FID Score(real data and model 2 synthetic data): {fid_value:.2f}")
"""


def evaluate_cvae(model, dataloader, device, num_classes=10):
    """
    Evaluate a CVAE model on a given dataloader.

    Args:
        model: The trained CVAE model.
        dataloader: PyTorch DataLoader for evaluation.
        device: torch.device('cuda') or torch.device('cpu').
        num_classes: Number of label classes (default=10).

    Returns:
        dict with average total loss, reconstruction loss (BCE), and KL divergence.
    """
    model.eval()
    total_loss = 0.0
    total_recon_loss = 0.0
    total_kl = 0.0
    num_samples = 0

    with torch.no_grad():
        for x, y in dataloader:
            x = x.view(-1, 784).to(device)
            y = F.one_hot(y, num_classes=num_classes).float().to(device)

            recon_x, mu, logvar = model(x, y)
            BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
            KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
            loss = BCE + KLD

            total_loss += loss.item()
            total_recon_loss += BCE.item()
            total_kl += KLD.item()
            num_samples += x.size(0)

    return {
        "avg_total_loss": total_loss / num_samples,
        "avg_recon_loss": total_recon_loss / num_samples,
        "avg_kl_divergence": total_kl / num_samples
    }

test_dataset = datasets.MNIST(root="./data", train=False, transform=transforms.ToTensor())
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

model1 = CVAE(latent_dim=latent_dim, label_dim=label_dim).to(device)
model1.load_state_dict(torch.load(f"model_saved/cvae_mnist_{sample_size}.pth"))
model1_score = evaluate_cvae(model1, test_loader, device)

model2 = CVAE(latent_dim=latent_dim, label_dim=label_dim).to(device)
model2.load_state_dict(torch.load(f"model_saved/cvae_mnist_filtered_synthetic_data_{sample_size}.pth"))
model2_score = evaluate_cvae(model2, test_loader, device)

# save the scores tp compare_result
compare_result = {
    "model1": model1_score,
    "model2": model2_score
}
import joblib
joblib.dump(compare_result, f"compare_result/cvae_mnist_{sample_size}_compare_result.pkl")

# Print the scores
print(f"Model 1 (original) scores: {model1_score}")
print(f"Model 2 (filtered synthetic) scores: {model2_score}")


Selected 30000 samples with p > 0.2
Loaded 30000 filtered synthetic samples
Epoch [1/200], Train Loss: 178.8523
Epoch [2/200], Train Loss: 132.3343
Epoch [3/200], Train Loss: 123.1486
Epoch [4/200], Train Loss: 119.1809
Epoch [5/200], Train Loss: 116.9844
Epoch [6/200], Train Loss: 115.4837
Epoch [7/200], Train Loss: 114.3535
Epoch [8/200], Train Loss: 113.4952
Epoch [9/200], Train Loss: 112.8259
Epoch [10/200], Train Loss: 112.2812
Epoch [11/200], Train Loss: 111.8458
Epoch [12/200], Train Loss: 111.4747
Epoch [13/200], Train Loss: 111.2282
Epoch [14/200], Train Loss: 110.9221
Epoch [15/200], Train Loss: 110.8027
Epoch [16/200], Train Loss: 110.5588
Epoch [17/200], Train Loss: 110.3810
Epoch [18/200], Train Loss: 110.2224
Epoch [19/200], Train Loss: 110.0759
Epoch [20/200], Train Loss: 109.9745
Epoch [21/200], Train Loss: 109.9492
Epoch [22/200], Train Loss: 109.8406
Epoch [23/200], Train Loss: 109.7792
Epoch [24/200], Train Loss: 109.6428
Epoch [25/200], Train Loss: 109.6304
Epoch [2

/tmp/ipykernel_1863339/1984115210.py:165: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model1.load_state_dict(torch.load(f"model_saved/cvae_mnist_{sample_size}.pth"))
/tmp/

Model 1 (original) scores: {'avg_total_loss': 106.72220415039062, 'avg_recon_loss': 84.90052819824218, 'avg_kl_divergence': 21.821676419067384}
Model 2 (filtered synthetic) scores: {'avg_total_loss': 108.00806572265626, 'avg_recon_loss': 87.45262963867188, 'avg_kl_divergence': 20.555436053466796}


In [1]:
import sys
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import os
import numpy as np
import random

sample_size = 5000
filter_threshold = 0.5

############################ real data training ############################
sys.path.append("/home/qiyuanliu/data_filter/Verified-Synthetic-Data/MNIST")
seed = 0
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

import torch.nn.functional as F
from cvae_model import CVAE, cvae_loss
from torch.utils.data import Subset

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
latent_dim = 20
label_dim = 10
batch_size = 128
epochs = 200
lr = 1e-3
# Load MNIST
transform = transforms.ToTensor()
full_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_dataset = Subset(full_dataset, range(sample_size))
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Initialize model
model = CVAE(latent_dim=latent_dim, label_dim=label_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# One-hot encoding helper
def one_hot(labels, num_classes=10):
    return F.one_hot(labels, num_classes).float()

best_train_loss = float('inf')
patience = 50
trigger_times = 0

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x = x.view(-1, 784).to(device)
        y = one_hot(y).to(device)

        optimizer.zero_grad()
        recon_x, mu, logvar = model(x, y)
        loss = cvae_loss(recon_x, x, mu, logvar)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {avg_loss:.4f}")

    # Early stopping based on training loss
    if avg_loss < best_train_loss:
        best_train_loss = avg_loss
        trigger_times = 0
    else:
        trigger_times += 1
        print(f"EarlyStopping counter: {trigger_times} out of {patience}")
        if trigger_times >= patience:
            print("Early stopping triggered.")
            break
os.chdir("/home/qiyuanliu/data_filter/Verified-Synthetic-Data/MNIST")  
# save the model to model_saved folder
torch.save(model.state_dict(), f"model_saved/cvae_mnist_{sample_size}.pth")
print(f"Model saved to model_saved/cvae_mnist_{sample_size}.pth")

############################ generate synthetic data ############################

model = CVAE(latent_dim=latent_dim, label_dim=label_dim)
model.load_state_dict(torch.load(f"model_saved/cvae_mnist_{sample_size}.pth"))
model.eval()

def generate_images_in_batches(model, total_samples, latent_dim, num_classes, batch_size=10000, device='cuda'):
    model.eval()
    generated_images = []
    all_labels = []

    for start in range(0, total_samples, batch_size):
        end = min(start + batch_size, total_samples)
        batch_size_actual = end - start

        # Generate z and y
        z = torch.randn(batch_size_actual, latent_dim).to(device)
        y = torch.arange(num_classes).repeat_interleave(total_samples // num_classes)[start:end]
        y_onehot = F.one_hot(y, num_classes=num_classes).float().to(device)

        with torch.no_grad():
            imgs = model.decode(z, y_onehot).view(-1, 1, 28, 28).cpu()
            generated_images.append(imgs)
            all_labels.append(y)

    images = torch.cat(generated_images, dim=0)
    labels = torch.cat(all_labels, dim=0)
    return images, labels

# large sample size for training
latent_dim = model.latent_dim
device = next(model.parameters()).device
gen_imgs_before_filter,y_before_filter = generate_images_in_batches(
    model=model,
    total_samples=6000000,
    latent_dim=latent_dim,
    num_classes=10,
    batch_size=10000,
    device=device
)

#save_path = f"data_saved/synthetic_mnist_cvae_{sample_size}.pt"
#torch.save({
#    'images': gen_imgs,    # Tensor [6000000, 1, 28, 28]
#    'labels': y            # Tensor [6000000]
#}, save_path)

# smaller sample size for evaluation
gen_imgs,y = generate_images_in_batches(
    model=model,
    total_samples=6000,
    latent_dim=latent_dim,
    num_classes=10,
    batch_size=10000,
    device=device
)

save_path = f"data_saved/synthetic_mnist_cvae_{sample_size}_2.pt"
torch.save({
    'images': gen_imgs,    # Tensor [6000, 1, 28, 28]
    'labels': y            # Tensor [6000]
}, save_path)

############################ filter synthetic data ############################

from discriminator import Discriminator
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
D = Discriminator().to(device)
D.load_state_dict(torch.load("model_saved/discriminator_mnist_cvae_more_epoch.pth"))
D.eval()

#data = torch.load(f"data_saved/synthetic_mnist_cvae_{sample_size}.pt")
#synthetic_images = data['images']  

synthetic_loader = DataLoader(gen_imgs_before_filter, batch_size=512)

all_probs = []

with torch.no_grad():
    for batch in synthetic_loader:
        batch = batch.to(device)
        probs = D(batch)  # [batch_size, 1], already sigmoid activated
        all_probs.append(probs.cpu())

all_probs = torch.cat(all_probs, dim=0)
# Flatten probs to shape [N]
probs = all_probs.squeeze(1)

# Load images and labels
images = gen_imgs_before_filter#data['images']      # [N, 1, 28, 28]
labels = y_before_filter #data['labels']      # [N]
# Create mask for p > filter_threshold
mask = probs > filter_threshold

# Apply mask
filtered_images = images[mask]
filtered_labels = labels[mask]

print(f"Selected {filtered_images.shape[0]} samples with p > {filter_threshold}")
# Save to file
torch.save({
    'images': filtered_images,
    'labels': filtered_labels
}, f"data_saved/synthetic_mnist_filtered_pgt{filter_threshold}_{sample_size}.pt")


############################ synthetic data retraining ############################

from torch.utils.data import TensorDataset

images = filtered_images  # shape: [N, 1, 28, 28]
labels = filtered_labels  # shape: [N]

print(f"Loaded {images.shape[0]} filtered synthetic samples")

# Preprocess: flatten images and convert labels to one-hot
images = images.view(-1, 784)  # flatten to [N, 784]

# Create dataset and dataloader
dataset = TensorDataset(images, labels)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialize model
model = CVAE(latent_dim=latent_dim, label_dim=label_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

best_train_loss = float('inf')
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x = x.view(-1, 784).to(device)
        y = one_hot(y).to(device)

        optimizer.zero_grad()
        recon_x, mu, logvar = model(x, y)
        loss = cvae_loss(recon_x, x, mu, logvar)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {avg_loss:.4f}")

    # Early stopping based on training loss
    if avg_loss < best_train_loss:
        best_train_loss = avg_loss
        trigger_times = 0
    else:
        trigger_times += 1
        print(f"EarlyStopping counter: {trigger_times} out of {patience}")
        if trigger_times >= patience:
            print("Early stopping triggered.")
            break

# save the model to model_saved folder
torch.save(model.state_dict(), f"model_saved/cvae_mnist_filtered_synthetic_data_{sample_size}.pth")

# use the new model to generate synthetic data for evaluation
model.eval()

n_per_class = 6000
num_classes = 10
total_samples = n_per_class * num_classes
latent_dim = model.latent_dim
device = next(model.parameters()).device

z = torch.randn(total_samples, latent_dim).to(device)
y = torch.arange(num_classes).repeat_interleave(n_per_class)
y_onehot = F.one_hot(y, num_classes=num_classes).float().to(device)

with torch.no_grad():
    gen_imgs = model.decode(z, y_onehot).view(-1, 1, 28, 28).cpu()  # shape: [60000, 1, 28, 28]

# save the generated images and labels
save_path = f"data_saved/synthetic_mnist_cvae_filtered_synthetic_model_generated_data_{sample_size}.pt"
torch.save({"images": gen_imgs, "labels": y}, save_path)

############################ Model Evaluation ############################
"""
# FID
from FID import calculate_fid_score_2

transform = transforms.ToTensor()

real_ds = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
### Synthetic dataset
synthetic = torch.load(f"data_saved/synthetic_mnist_cvae_{sample_size}_2.pt")
synthetic_ds = TensorDataset(synthetic['images'], torch.zeros(len(synthetic['images'])))
fid_value = calculate_fid_score_2(real_ds, synthetic_ds)
print(f"FID Score(real data and synthetic data): {fid_value:.2f}")

## filtered synthetic data
synthetic = torch.load(f"data_saved/synthetic_mnist_filtered_pgt{filter_threshold}_{sample_size}.pt")
synthetic_ds = TensorDataset(synthetic['images'], torch.zeros(len(synthetic['images'])))
fid_value = calculate_fid_score_2(real_ds, synthetic_ds)
print(f"FID Score(real data and filtered synthetic data): {fid_value:.2f}")

# Synthetic dataset
synthetic = torch.load(f"data_saved/synthetic_mnist_cvae_filtered_synthetic_model_generated_data_{sample_size}.pt")
synthetic_ds = TensorDataset(synthetic['images'], torch.zeros(len(synthetic['images'])))
fid_value = calculate_fid_score_2(real_ds, synthetic_ds)
print(f"FID Score(real data and model 2 synthetic data): {fid_value:.2f}")
"""


def evaluate_cvae(model, dataloader, device, num_classes=10):
    """
    Evaluate a CVAE model on a given dataloader.

    Args:
        model: The trained CVAE model.
        dataloader: PyTorch DataLoader for evaluation.
        device: torch.device('cuda') or torch.device('cpu').
        num_classes: Number of label classes (default=10).

    Returns:
        dict with average total loss, reconstruction loss (BCE), and KL divergence.
    """
    model.eval()
    total_loss = 0.0
    total_recon_loss = 0.0
    total_kl = 0.0
    num_samples = 0

    with torch.no_grad():
        for x, y in dataloader:
            x = x.view(-1, 784).to(device)
            y = F.one_hot(y, num_classes=num_classes).float().to(device)

            recon_x, mu, logvar = model(x, y)
            BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
            KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
            loss = BCE + KLD

            total_loss += loss.item()
            total_recon_loss += BCE.item()
            total_kl += KLD.item()
            num_samples += x.size(0)

    return {
        "avg_total_loss": total_loss / num_samples,
        "avg_recon_loss": total_recon_loss / num_samples,
        "avg_kl_divergence": total_kl / num_samples
    }

test_dataset = datasets.MNIST(root="./data", train=False, transform=transforms.ToTensor())
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

model1 = CVAE(latent_dim=latent_dim, label_dim=label_dim).to(device)
model1.load_state_dict(torch.load(f"model_saved/cvae_mnist_{sample_size}.pth"))
model1_score = evaluate_cvae(model1, test_loader, device)

model2 = CVAE(latent_dim=latent_dim, label_dim=label_dim).to(device)
model2.load_state_dict(torch.load(f"model_saved/cvae_mnist_filtered_synthetic_data_{sample_size}.pth"))
model2_score = evaluate_cvae(model2, test_loader, device)

# save the scores tp compare_result
compare_result = {
    "model1": model1_score,
    "model2": model2_score
}
import joblib
joblib.dump(compare_result, f"compare_result/cvae_mnist_{sample_size}_compare_result.pkl")

# Print the scores
print(f"Model 1 (original) scores: {model1_score}")
print(f"Model 2 (filtered synthetic) scores: {model2_score}")


Epoch [1/200], Train Loss: 289.2108
Epoch [2/200], Train Loss: 203.2987
Epoch [3/200], Train Loss: 180.5909
Epoch [4/200], Train Loss: 163.6566
Epoch [5/200], Train Loss: 153.5960
Epoch [6/200], Train Loss: 147.1924
Epoch [7/200], Train Loss: 142.2007
Epoch [8/200], Train Loss: 138.1660
Epoch [9/200], Train Loss: 135.0459
Epoch [10/200], Train Loss: 131.6477
Epoch [11/200], Train Loss: 128.9189
Epoch [12/200], Train Loss: 126.7101
Epoch [13/200], Train Loss: 124.5109
Epoch [14/200], Train Loss: 122.8315
Epoch [15/200], Train Loss: 121.3991
Epoch [16/200], Train Loss: 120.2449
Epoch [17/200], Train Loss: 119.0302
Epoch [18/200], Train Loss: 117.9980
Epoch [19/200], Train Loss: 116.8348
Epoch [20/200], Train Loss: 115.9087
Epoch [21/200], Train Loss: 115.1816
Epoch [22/200], Train Loss: 114.4127
Epoch [23/200], Train Loss: 113.7345
Epoch [24/200], Train Loss: 112.9594
Epoch [25/200], Train Loss: 112.5227
Epoch [26/200], Train Loss: 111.9417
Epoch [27/200], Train Loss: 111.3447
Epoch [28/

/tmp/ipykernel_2336875/3830116846.py:90: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"model_saved/cvae_mnist_{sample_size}.pth"))
/tmp/ip

Selected 51531 samples with p > 0.5
Loaded 51531 filtered synthetic samples
Epoch [1/200], Train Loss: 154.8993
Epoch [2/200], Train Loss: 113.8144
Epoch [3/200], Train Loss: 106.6459
Epoch [4/200], Train Loss: 103.3960
Epoch [5/200], Train Loss: 101.5282
Epoch [6/200], Train Loss: 100.3566
Epoch [7/200], Train Loss: 99.4634
Epoch [8/200], Train Loss: 98.7495
Epoch [9/200], Train Loss: 98.2852
Epoch [10/200], Train Loss: 97.8172
Epoch [11/200], Train Loss: 97.5329
Epoch [12/200], Train Loss: 97.3002
Epoch [13/200], Train Loss: 97.0653
Epoch [14/200], Train Loss: 96.8473
Epoch [15/200], Train Loss: 96.6970
Epoch [16/200], Train Loss: 96.6002
Epoch [17/200], Train Loss: 96.4791
Epoch [18/200], Train Loss: 96.3362
Epoch [19/200], Train Loss: 96.2551
Epoch [20/200], Train Loss: 96.2335
Epoch [21/200], Train Loss: 96.1020
Epoch [22/200], Train Loss: 96.0122
Epoch [23/200], Train Loss: 95.9213
Epoch [24/200], Train Loss: 95.9061
Epoch [25/200], Train Loss: 95.8272
Epoch [26/200], Train Loss:

/tmp/ipykernel_2336875/3830116846.py:339: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model1.load_state_dict(torch.load(f"model_saved/cvae_mnist_{sample_size}.pth"))
/tmp/

Model 1 (original) scores: {'avg_total_loss': 107.12878720703125, 'avg_recon_loss': 85.23452104492188, 'avg_kl_divergence': 21.89426642456055}
Model 2 (filtered synthetic) scores: {'avg_total_loss': 110.62587829589843, 'avg_recon_loss': 89.28567764892578, 'avg_kl_divergence': 21.340200604248047}


In [7]:
synthetic = torch.load(f"data_saved/synthetic_mnist_cvae_filtered_synthetic_model_generated_data_{sample_size}.pt")
synthetic_ds = TensorDataset(synthetic['images'], torch.zeros(len(synthetic['images'])))
fid_value = calculate_fid_score_2(real_ds, synthetic_ds)
print(f"FID Score(real data and model 2 synthetic data): {fid_value:.2f}")

/tmp/ipykernel_1863339/1644706098.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  synthetic = torch.load(f"data_saved/synthetic_mnist_cvae_filtered_synthetic_model_gener

FID Score(real data and model 2 synthetic data): 140.14


In [6]:
# save the model to model_saved folder
#torch.save(model.state_dict(), f"model_saved/cvae_mnist_filtered_synthetic_data_{sample_size}.pth")
model = CVAE(latent_dim=latent_dim, label_dim=label_dim).to(device)
model.load_state_dict(torch.load(f"model_saved/cvae_mnist_filtered_synthetic_data_{sample_size}.pth"))
# use the new model to generate synthetic data for evaluation
model.eval()

n_per_class = 6000
num_classes = 10
total_samples = n_per_class * num_classes
latent_dim = model.latent_dim
device = next(model.parameters()).device

z = torch.randn(total_samples, latent_dim).to(device)
y = torch.arange(num_classes).repeat_interleave(n_per_class)
y_onehot = F.one_hot(y, num_classes=num_classes).float().to(device)

with torch.no_grad():
    gen_imgs = model.decode(z, y_onehot).view(-1, 1, 28, 28).cpu()  # shape: [60000, 1, 28, 28]

# save the generated images and labels
save_path = f"data_saved/synthetic_mnist_cvae_filtered_synthetic_model_generated_data_{sample_size}.pt"
torch.save({"images": gen_imgs, "labels": y}, save_path)

/tmp/ipykernel_1863339/2097051580.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"model_saved/cvae_mnist_filtered_synthetic_data_{samp

In [8]:
from torch.utils.data import TensorDataset

images = filtered_images  # shape: [N, 1, 28, 28]
labels = filtered_labels  # shape: [N]

print(f"Loaded {images.shape[0]} filtered synthetic samples")

# Preprocess: flatten images and convert labels to one-hot
images = images.view(-1, 784)  # flatten to [N, 784]

# Create dataset and dataloader
dataset = TensorDataset(images, labels)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialize model
model = CVAE(latent_dim=latent_dim, label_dim=label_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
patience =50
best_train_loss = float('inf')
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x = x.view(-1, 784).to(device)
        y = one_hot(y).to(device)

        optimizer.zero_grad()
        recon_x, mu, logvar = model(x, y)
        loss = cvae_loss(recon_x, x, mu, logvar)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {avg_loss:.4f}")

    # Early stopping based on training loss
    if avg_loss < best_train_loss:
        best_train_loss = avg_loss
        trigger_times = 0
    else:
        trigger_times += 1
        print(f"EarlyStopping counter: {trigger_times} out of {patience}")
        if trigger_times >= patience:
            print("Early stopping triggered.")
            break

# save the model to model_saved folder
torch.save(model.state_dict(), f"model_saved/cvae_mnist_filtered_synthetic_data_{sample_size}.pth")

# use the new model to generate synthetic data for evaluation
model.eval()

n_per_class = 6000
num_classes = 10
total_samples = n_per_class * num_classes
latent_dim = model.latent_dim
device = next(model.parameters()).device

z = torch.randn(total_samples, latent_dim).to(device)
y = torch.arange(num_classes).repeat_interleave(n_per_class)
y_onehot = F.one_hot(y, num_classes=num_classes).float().to(device)

with torch.no_grad():
    gen_imgs = model.decode(z, y_onehot).view(-1, 1, 28, 28).cpu()  # shape: [60000, 1, 28, 28]

# save the generated images and labels
save_path = f"data_saved/synthetic_mnist_cvae_filtered_synthetic_model_generated_data_{sample_size}.pt"
torch.save({"images": gen_imgs, "labels": y}, save_path)


Loaded 61090 filtered synthetic samples
Epoch [1/200], Train Loss: 143.6816
Epoch [2/200], Train Loss: 106.6899
Epoch [3/200], Train Loss: 100.9093
Epoch [4/200], Train Loss: 98.2640
Epoch [5/200], Train Loss: 96.7571
Epoch [6/200], Train Loss: 95.8245
Epoch [7/200], Train Loss: 95.1488
Epoch [8/200], Train Loss: 94.6389
Epoch [9/200], Train Loss: 94.2613
Epoch [10/200], Train Loss: 93.9233
Epoch [11/200], Train Loss: 93.6934
Epoch [12/200], Train Loss: 93.4911
Epoch [13/200], Train Loss: 93.3289
Epoch [14/200], Train Loss: 93.1535
Epoch [15/200], Train Loss: 93.0498
Epoch [16/200], Train Loss: 92.9056
Epoch [17/200], Train Loss: 92.8120
Epoch [18/200], Train Loss: 92.7578
Epoch [19/200], Train Loss: 92.5851
Epoch [20/200], Train Loss: 92.5783
Epoch [21/200], Train Loss: 92.4491
Epoch [22/200], Train Loss: 92.4167
Epoch [23/200], Train Loss: 92.3571
Epoch [24/200], Train Loss: 92.3295
Epoch [25/200], Train Loss: 92.2657
Epoch [26/200], Train Loss: 92.2079
Epoch [27/200], Train Loss: 92

In [ ]:
synthetic = torch.load(f"data_saved/synthetic_mnist_cvae_filtered_synthetic_model_generated_data_{sample_size}.pt")
synthetic_ds = TensorDataset(synthetic['images'], torch.zeros(len(synthetic['images'])))
fid_value = calculate_fid_score_2(real_ds, synthetic_ds)
print(f"FID Score(real data and model 2 synthetic data): {fid_value:.2f}")

/tmp/ipykernel_1863339/1644706098.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  synthetic = torch.load(f"data_saved/synthetic_mnist_cvae_filtered_synthetic_model_gener

FID Score(real data and model 2 synthetic data): 138.28


: 